# V6.5-15 — Dataset 5 class cho YOLO (bỏ `insulator` khi train)
Audit nhãn cho thấy các nguồn ảnh lỗi (insulator_broken / flashover / broken_strand) **không gán nhãn `insulator`** dù ảnh có sứ → model học "sứ là nền" ở nguồn này nhưng phải phát hiện sứ ở nguồn 1280×720. Mục tiêu triển khai chỉ cảnh báo lỗi nên bỏ `insulator` khỏi tập train YOLO.

- **Không sửa dataset gốc** `v6p5_roboflow_ablation`: ảnh là symlink từng file về bản gốc, chỉ viết labels mới + `data.yaml` mới.
- Class map: `insulator` bị bỏ; 1..5 → 0..4. Ảnh chỉ có `insulator` thành ảnh nền (negative), vẫn giữ để giảm báo động giả.
- Chỉ dùng CPU. Chạy được lại: build vào thư mục tạm rồi đổi tên; đã có output thì dừng trừ khi `OVERWRITE=True`.
- Nhãn `insulator` gốc vẫn còn nguyên để sau này làm lại đúng (pseudo-label + duyệt tay) cho tầng RT-DETR.

In [ ]:
# CONFIG
from pathlib import Path
import json, os, shutil, hashlib, collections, random
from datetime import datetime, timezone
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "Uav-Powerline-Defect-Detection"]
PROJECT = next(p for p in candidates if (p / "pyproject.toml").is_file() and (p / "legacy/notebooks/v6p5").is_dir())
SRC = PROJECT.parent / "datasets/v6p5_roboflow_ablation"
DST = PROJECT.parent / "datasets/v6p5_5cls_clean"
SRC_NAMES = ["insulator", "insulator_broken", "insulator_flashover", "bird_nest", "foreign_object", "broken_strand"]
DROP = {"insulator"}
DST_NAMES = [n for n in SRC_NAMES if n not in DROP]
CLASS_MAP = {SRC_NAMES.index(n): i for i, n in enumerate(DST_NAMES)}  # src id -> dst id
SPLITS = ["train", "val"]
OVERWRITE = False
for required in [SRC / "data.yaml", *(SRC / "images" / s for s in SPLITS), *(SRC / "labels" / s for s in SPLITS)]:
    if not required.exists(): raise FileNotFoundError(required)
print("SRC:", SRC)
print("DST:", DST)
print("CLASS_MAP:", {SRC_NAMES[k]: f"{k}->{v}" for k, v in CLASS_MAP.items()}, "| drop:", sorted(DROP))

SRC: /teamspace/studios/this_studio/datasets/v6p5_roboflow_ablation
DST: /teamspace/studios/this_studio/datasets/v6p5_roboflow_5cls
CLASS_MAP: {'insulator_broken': '1->0', 'insulator_flashover': '2->1', 'bird_nest': '3->2', 'foreign_object': '4->3', 'broken_strand': '5->4'} | drop: ['insulator']


In [2]:
# 1. KIỂM TRA TAXONOMY NGUỒN
import yaml
cfg = yaml.safe_load((SRC / "data.yaml").read_text())
names = cfg["names"]
names = [names[k] for k in sorted(names, key=int)] if isinstance(names, dict) else list(names)
if names != SRC_NAMES:
    raise ValueError(f"Taxonomy nguồn khác dự kiến: {names}")
if DST.exists() and not OVERWRITE:
    raise FileExistsError(f"{DST} đã tồn tại. Đặt OVERWRITE=True để build lại.")
print("Taxonomy OK")

Taxonomy OK


In [3]:
# 2. BUILD: symlink ảnh + viết labels đã remap (vào thư mục tạm rồi đổi tên)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
tmp = DST.with_name(DST.name + ".tmp")
if tmp.exists(): shutil.rmtree(tmp)
stats = {}
for split in SPLITS:
    (tmp / "images" / split).mkdir(parents=True)
    (tmp / "labels" / split).mkdir(parents=True)
    s = collections.Counter()
    for img in sorted((SRC / "images" / split).iterdir()):
        if img.suffix.lower() not in IMG_EXT: continue
        (tmp / "images" / split / img.name).symlink_to(img.resolve())
        src_lbl = SRC / "labels" / split / (img.stem + ".txt")
        out = []
        if src_lbl.is_file():
            for line in src_lbl.read_text().splitlines():
                p = line.split()
                if not p: continue
                c = int(p[0])
                if c not in CLASS_MAP and SRC_NAMES[c] not in DROP:
                    raise ValueError(f"Class id lạ {c} trong {src_lbl}")
                s[f"src_{SRC_NAMES[c]}"] += 1
                if c in CLASS_MAP:
                    out.append(" ".join([str(CLASS_MAP[c]), *p[1:]]))
                    s[f"dst_{DST_NAMES[CLASS_MAP[c]]}"] += 1
        had = src_lbl.is_file() and src_lbl.stat().st_size > 0
        s["images"] += 1
        s["bg_before"] += not had
        s["bg_after"] += not out
        s["became_bg"] += had and not out
        (tmp / "labels" / split / (img.stem + ".txt")).write_text("\n".join(out) + ("\n" if out else ""))
    stats[split] = dict(s)
(tmp / "data.yaml").write_text(yaml.safe_dump({
    "path": str(DST), "train": "images/train", "val": "images/val",
    "names": {i: n for i, n in enumerate(DST_NAMES)},
}, sort_keys=False, allow_unicode=True))
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "source": str(SRC), "dropped": sorted(DROP),
    "class_map": {SRC_NAMES[k]: DST_NAMES[v] for k, v in CLASS_MAP.items()},
    "source_data_yaml_sha256": hashlib.sha256((SRC / "data.yaml").read_bytes()).hexdigest(),
    "stats": stats,
    "reason": "Nguồn ảnh lỗi không gán nhãn insulator -> xung đột nhãn; mục tiêu triển khai chỉ cảnh báo lỗi.",
}
(tmp / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
if DST.exists(): shutil.rmtree(DST)
tmp.rename(DST)
print("Built:", DST)

Built: /teamspace/studios/this_studio/datasets/v6p5_roboflow_5cls


In [4]:
# 3. TỔNG KẾT
import pandas as pd
rows = []
for split, s in stats.items():
    for n in SRC_NAMES:
        rows.append({"split": split, "class": n, "src": s.get(f"src_{n}", 0),
                     "dst": s.get(f"dst_{n}", 0) if n not in DROP else 0})
display(pd.DataFrame(rows).pivot_table(index="class", columns="split", values=["src", "dst"]).astype(int))
for split, s in stats.items():
    print(f"{split}: {s['images']} ảnh | nền trước={s['bg_before']} ({s['bg_before']/s['images']:.1%}) "
          f"-> sau={s['bg_after']} ({s['bg_after']/s['images']:.1%}) | ảnh mới thành nền={s['became_bg']}")

dst        src     
split               train  val train  val
class                                    
bird_nest            6136  821  6136  821
broken_strand        2030  252  2030  252
foreign_object       1491   98  1491   98
insulator               0    0  8665  778
insulator_broken     6236  791  6236  791
insulator_flashover  1249  163  1249  163

train: 18334 ảnh | nền trước=1437 (7.8%) -> sau=3733 (20.4%) | ảnh mới thành nền=2296
val: 2272 ảnh | nền trước=179 (7.9%) -> sau=481 (21.2%) | ảnh mới thành nền=302


In [5]:
# 4. VERIFY: Ultralytics đọc được dataset mới, label cache khớp
from ultralytics.data.utils import check_det_dataset
from ultralytics.data import YOLODataset
data = check_det_dataset(str(DST / "data.yaml"))
assert list(data["names"].values()) == DST_NAMES, data["names"]
for split in SPLITS:
    ds = YOLODataset(img_path=str(DST / "images" / split), data=data, augment=False, imgsz=640)
    n_inst = sum(len(l["cls"]) for l in ds.labels)
    expect = sum(stats[split].get(f"dst_{n}", 0) for n in DST_NAMES)
    assert len(ds.labels) == stats[split]["images"], (len(ds.labels), stats[split]["images"])
    assert n_inst == expect, (n_inst, expect)
    assert max((int(c) for l in ds.labels for c in l["cls"].flatten()), default=0) < len(DST_NAMES)
    print(f"{split}: {len(ds.labels)} ảnh, {n_inst} instance — OK")
# Dataset gốc không bị đụng tới
assert yaml.safe_load((SRC / "data.yaml").read_text())["names"] == cfg["names"]
print("DATA_YAML cho notebook train:", DST / "data.yaml")
print("NAMES =", DST_NAMES)

Fast image access ✅ (ping: 0.0±0.0 ms, read: 746.0±538.5 MB/s, size: 307.7 KB)



Scanning /teamspace/studios/this_studio/datasets/v6p5_roboflow_5cls/labels/train... 34 images, 5 backgrounds, 0 corrupt: 0% ──────────── 34/18334 101.6it/s 0.1s<3:00


Scanning /teamspace/studios/this_studio/datasets/v6p5_roboflow_5cls/labels/train... 149 images, 36 backgrounds, 0 corrupt: 0% ──────────── 149/18334 415.5it/s 0.2s<43.8s


Scanning /teamspace/studios/this_studio/datasets/v6p5_roboflow_5cls/labels/train... 299 images, 73 backgrounds, 0 corrupt: 1% ──────────── 299/18334 739.4it/s 0.3s<24.4s


Scanning /teamspace/studios/this_studio/datasets/v6p5_roboflow_5cls/labels/train... 453 images, 104 backgrounds, 0 corrupt: 2% ──────────── 453/18334 976.7it/s 0.4s<18.3s

...


Scanning /teamspace/studios/this_studio/datasets/v6p5_roboflow_5cls/labels/val... 2080 images, 445 backgrounds, 0 corrupt: 91% ━━━━━━━━━━╸─ 2080/2272 1.4Kit/s 1.5s<0.1s


Scanning /teamspace/studios/this_studio/datasets/v6p5_roboflow_5cls/labels/val... 2246 images, 478 backgrounds, 0 corrupt: 98% ━━━━━━━━━━━╸ 2246/2272 1.5Kit/s 1.6s<0.0s


Scanning /teamspace/studios/this_studio/datasets/v6p5_roboflow_5cls/labels/val... 2272 images, 481 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2272/2272 1.4Kit/s 1.6s

New cache created: /teamspace/studios/this_studio/datasets/v6p5_roboflow_5cls/labels/val.cache


val: 2272 ảnh, 2125 instance — OK
DATA_YAML cho notebook train: /teamspace/studios/this_studio/datasets/v6p5_roboflow_5cls/data.yaml
NAMES = ['insulator_broken', 'insulator_flashover', 'bird_nest', 'foreign_object', 'broken_strand']


**Dùng cho train:** trong notebook train đổi `DATASET = PROJECT.parent / "datasets/v6p5_roboflow_5cls"` và `NAMES` thành 5 class ở trên. mAP sau khi bỏ `insulator` **không so trực tiếp** với các run 6 class (mAP "all" cũ được kéo lên bởi insulator AP50≈0.94); so theo AP từng class lỗi.